# 04 — Modelo CNN com Imagens
### GalaxyNet — Classificador Morfológico de Galáxias

Rede Neural Convolucional (CNN) usando imagens de galáxias $(64 \times 64 \times 3)$ nas bandas $g$, $r$, $i$. Arquitetura: 4 blocos Conv2D(3x3) → BN → MaxPool → Dropout, seguidos de Dense(512) → Softmax(3).

Data augmentation com rotação 360°, flips e zoom para compensar o dataset pequeno (156 amostras).

In [ ]:
import sys
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

def _find_src_dir():
    current = os.path.abspath(os.getcwd())
    for _ in range(5):
        candidate = os.path.join(current, 'src')
        if os.path.isdir(candidate):
            return candidate
        current = os.path.dirname(current)
    raise RuntimeError("Diretório src/ não encontrado.")

src_path = _find_src_dir()
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

from preprocessing import load_preprocessed_images, load_preprocessed_tabular
from models import create_cnn_model
from evaluation import evaluate_galaxy_classifier

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# Caminhos
_project_root = os.path.dirname(src_path)
PROCESSED_DIR = os.path.join(_project_root, 'data', 'processed')
MODELS_DIR    = os.path.join(_project_root, 'models')
REPORTS_DIR   = os.path.join(_project_root, 'reports', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"TensorFlow    : {tf.__version__}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"MODELS_DIR    : {MODELS_DIR}")

---
## 1. Carregamento dos Dados

Carrega as imagens pré-processadas e os rótulos. As imagens e os dados tabulares são alinhados por `objid` para garantir correspondência correta entre imagens e labels.

In [ ]:
# Carregar imagens pré-processadas
X_images, img_objids = load_preprocessed_images(PROCESSED_DIR)

# Carregar labels tabulares (mesma ordem de objids)
_, y_labels, tab_objids, _ = load_preprocessed_tabular(PROCESSED_DIR)

# Alinhar imagens e labels por objid
# Criar mapeamento objid → label
objid_to_label = dict(zip(tab_objids, y_labels))

# Filtrar apenas imagens que têm label correspondente e manter alinhamento
mask = np.array([oid in objid_to_label for oid in img_objids])
X_images = X_images[mask]
img_objids = img_objids[mask]
y_labels_aligned = np.array([objid_to_label[oid] for oid in img_objids])

print(f"\nX_images        : {X_images.shape}  dtype={X_images.dtype}")
print(f"y_labels        : {y_labels_aligned.shape}")
print(f"Valores em [0,1]: min={X_images.min():.4f}, max={X_images.max():.4f}")

print("\nDistribuição de classes:")
unique, counts = np.unique(y_labels_aligned, return_counts=True)
for cls, n in zip(unique, counts):
    print(f"  {cls:<12}: {n:>4} ({100*n/len(y_labels_aligned):.1f}%)")

---
## 2. Encoding de Labels e Split Train/Val/Test

Reutiliza o `LabelEncoder` salvo pelo notebook 03 para manter a mesma codificação entre modelos.

In [ ]:
# Carregar LabelEncoder do notebook 03 para consistência
le_path = os.path.join(MODELS_DIR, 'label_encoder.pkl')
if os.path.exists(le_path):
    with open(le_path, 'rb') as f:
        label_encoder = pickle.load(f)
    print(f"LabelEncoder carregado de: {le_path}")
else:
    label_encoder = LabelEncoder()
    label_encoder.fit(y_labels_aligned)
    print("LabelEncoder ajustado localmente (label_encoder.pkl não encontrado)")

y_encoded = label_encoder.transform(y_labels_aligned)
y_one_hot = to_categorical(y_encoded)

num_classes = y_one_hot.shape[1]
print(f"Classes: {list(label_encoder.classes_)} → {list(range(num_classes))}")

# Split estratificado: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp, objids_train, objids_temp = train_test_split(
    X_images, y_one_hot, img_objids,
    test_size=0.30, random_state=SEED, stratify=y_encoded
)

y_temp_encoded = np.argmax(y_temp, axis=1)
X_val, X_test, y_val, y_test, objids_val, objids_test = train_test_split(
    X_temp, y_temp, objids_temp,
    test_size=0.50, random_state=SEED, stratify=y_temp_encoded
)

print(f"\nTrain : {X_train.shape[0]} amostras — shape {X_train.shape}")
print(f"Val   : {X_val.shape[0]} amostras")
print(f"Test  : {X_test.shape[0]} amostras")

for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    labels = label_encoder.inverse_transform(np.argmax(y_split, axis=1))
    unique, counts = np.unique(labels, return_counts=True)
    dist = ', '.join(f"{c}:{n}" for c, n in zip(unique, counts))
    print(f"  {name}: {dist}")

---
## 3. Data Augmentation e Class Weights

Galáxias não possuem orientação preferencial no céu, o que justifica:
- **Rotação 360°**: qualquer ângulo é fisicamente válido
- **Flips horizontal e vertical**: simetria do céu
- **Zoom 0.8–1.2**: simula diferentes distâncias angulares
- **Brightness 0.8–1.2**: simula variações de exposição

O `fill_mode='constant'` com `cval=0.0` preenche bordas com zero (céu escuro).

In [ ]:
# Data augmentation — apenas no treino
datagen = ImageDataGenerator(
    rotation_range=360,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    fill_mode='constant',
    cval=0.0,
)

# Visualizar exemplos de data augmentation
sample_img = X_train[0:1]
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(np.stack([sample_img[0,:,:,1], sample_img[0,:,:,0], sample_img[0,:,:,2]], axis=-1),
               origin='lower')
axes[0].set_title('Original')
axes[0].axis('off')

for i, batch in enumerate(datagen.flow(sample_img, batch_size=1)):
    if i >= 5:
        break
    img = batch[0]
    rgb = np.stack([img[:,:,1], img[:,:,0], img[:,:,2]], axis=-1)
    axes[i+1].imshow(np.clip(rgb, 0, 1), origin='lower')
    axes[i+1].set_title(f'Aug {i+1}')
    axes[i+1].axis('off')

fig.suptitle('Data Augmentation — exemplos de transformações', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig18_cnn_data_augmentation.png'), bbox_inches='tight')
plt.show()
print('Figura salva: fig18_cnn_data_augmentation.png')

# Class weights
y_train_integers = np.argmax(y_train, axis=1)
class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_integers),
    y=y_train_integers,
)
class_weight_dict = dict(enumerate(class_weights_array))

print("\nClass weights (balanced):")
for idx, weight in class_weight_dict.items():
    cls_name = label_encoder.inverse_transform([idx])[0]
    n_train = (y_train_integers == idx).sum()
    print(f"  {cls_name:<12} (idx={idx}): weight={weight:.3f}  (n_train={n_train})")

---
## 4. Criação do Modelo CNN

In [ ]:
input_shape = X_train.shape[1:]  # (64, 64, 3)
cnn_model = create_cnn_model(input_shape, num_classes)
cnn_model.summary()

---
## 5. Treinamento

O treinamento usa `datagen.flow()` para aplicar data augmentation em tempo real a cada batch. A validação usa os dados originais (sem augmentation).

- **EarlyStopping**: patience=15, restaura melhores pesos
- **ReduceLROnPlateau**: reduz LR pela metade se val_loss estagnar por 7 épocas

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=7, min_lr=1e-7, verbose=1
    ),
]

history = cnn_model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    validation_data=(X_val, y_val),
    epochs=100,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

print(f"\nTreinamento finalizado em {len(history.history['loss'])} épocas.")

---
## 6. Curvas de Treinamento (Loss e Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss — CNN', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (Categorical Crossentropy)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy — CNN', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'fig19_cnn_training_curves.png'),
            bbox_inches='tight')
plt.show()
print('Figura salva: fig19_cnn_training_curves.png')

---
## 7. Avaliação no Conjunto de Teste

In [ ]:
loss, accuracy = cnn_model.evaluate(X_test, y_test, verbose=0)
print(f"CNN Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

eval_results = evaluate_galaxy_classifier(
    model=cnn_model,
    X_test_data=X_test,
    y_test_one_hot=y_test,
    label_encoder=label_encoder,
    model_type='CNN',
    save_dir=REPORTS_DIR,
)

---
## 8. Salvar Modelo

In [ ]:
# Salvar modelo treinado
model_path = os.path.join(MODELS_DIR, 'cnn_galaxy_classifier.h5')
cnn_model.save(model_path)
print(f"Modelo salvo em: {model_path}")

# Salvar dados de teste para comparação posterior entre modelos
test_artifacts = {
    'X_test': X_test,
    'y_test': y_test,
    'objids_test': objids_test,
}
test_path = os.path.join(MODELS_DIR, 'cnn_test_data.pkl')
with open(test_path, 'wb') as f:
    pickle.dump(test_artifacts, f)
print(f"Dados de teste salvos em: {test_path}")

---
## 9. Resumo

In [ ]:
print("=" * 55)
print("RESUMO — MODELO CNN COM IMAGENS")
print("=" * 55)
print(f"Input shape          : {input_shape}")
print(f"Classes              : {list(label_encoder.classes_)}")
print(f"Amostras (train/val/test): {X_train.shape[0]}/{X_val.shape[0]}/{X_test.shape[0]}")
print(f"Data augmentation    : rotation=360°, flips, zoom=0.2, brightness=[0.8,1.2]")
print(f"Épocas treinadas     : {len(history.history['loss'])}")
print(f"Test Loss            : {loss:.4f}")
print(f"Test Accuracy        : {accuracy:.4f}")
print()
print("Métricas Científicas:")
print(eval_results['scientific_metrics'].to_string(index=False))
print()
print("Artefatos salvos:")
print(f"  1. {model_path}")
print(f"  2. reports/figures/fig18_cnn_data_augmentation.png")
print(f"  3. reports/figures/fig19_cnn_training_curves.png")
print(f"  4. reports/figures/fig17_cnn_confusion_matrix.png")